# MovieDB Data Cleaning and Visualization Project

**Project type:** Data Cleaning and Visualization  
**Dataset:** Real MovieDB CSV dataset  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Goal:** Convert raw movie data into a clean dataset, charts, insights, recommendations, and a visual report.

## 1. Project Title

MovieDB Data Cleaning and Visualization Project

## Project Overview

Raw datasets usually need cleaning before they can be trusted. In this project, I cleaned the MovieDB CSV, fixed missing and incorrect values, created useful features, and built charts to understand the movie collection.

This kind of analysis is useful for movie websites and streaming platforms because it helps compare genres, track audience interest, and support recommendation ideas.

## 2. Problem Statement

The dataset contains movie details such as release date, title, overview, popularity, vote count, rating, language, genre, and poster URL. The goal is to clean the raw file and use it to answer practical questions about movie performance and audience interest.

The workflow covers loading, cleaning, feature engineering, exploratory analysis, visualization, insights, and final recommendations.

## 3. Import Libraries

Set up the main Python libraries used for cleaning, analysis, and visualization.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 4. Load Dataset

The CSV is loaded with a fallback method because long text fields can sometimes break the default reader. This keeps the project stable on real data.

In [ ]:
raw_path = PROJECT_ROOT / "data" / "raw" / "mymoviedb_raw.csv"

def load_movie_data(path):
    try:
        df = pd.read_csv(path)
        method = "default"
    except Exception:
        df = pd.read_csv(path, engine="python", on_bad_lines="skip")
        method = "python_engine"
    return df, method

raw_df, load_method = load_movie_data(raw_path)
print("Loaded with:", load_method)
print("Shape:", raw_df.shape)
raw_df.head()

## 5. Dataset Overview

Before cleaning, we inspect rows, columns, data types, missing values, duplicates, and sample records.

In [ ]:
print("Rows and columns:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head())
display(raw_df.info())
display(raw_df.describe(include="all").T)

In [ ]:
print("Missing values:")
display(raw_df.isna().sum().sort_values(ascending=False))

print("Duplicate rows:", raw_df.duplicated().sum())

## 6. Data Cleaning

Clean the dataset so the final analysis is based on reliable values.

In [ ]:
def standardize_column_names(df):
    cleaned = df.copy()
    cleaned.columns = (
        cleaned.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned

df = standardize_column_names(raw_df)
df.columns.tolist()

In [ ]:
def clean_text_columns(df):
    cleaned = df.copy()
    placeholders = {"": np.nan, "nan": np.nan, "none": np.nan, "null": np.nan, "n/a": np.nan, "na": np.nan, "unknown": np.nan}
    for column in cleaned.select_dtypes(include=["object", "string"]).columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()
        cleaned[column] = cleaned[column].replace(placeholders)
    return cleaned

df = clean_text_columns(df)
print("Text columns cleaned.")

In [ ]:
def fix_data_types(df):
    cleaned = df.copy()
    cleaned["release_date"] = pd.to_datetime(cleaned["release_date"], errors="coerce", format="mixed")
    for column in ["popularity", "vote_count", "vote_average"]:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
    return cleaned

df = fix_data_types(df)
display(df.dtypes)

In [ ]:
before_rows = len(df)
df = df[df["release_date"].notna() & df["title"].notna()].copy()
print("Unusable rows removed:", before_rows - len(df))

duplicate_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop_duplicates(subset=["title", "release_date"]).reset_index(drop=True)
print("Exact duplicates before removal:", duplicate_before)
print("Duplicates after removal:", df.duplicated().sum())

### Missing Value Handling

Fill missing text values with clear labels and numeric values with the median.

In [ ]:
df["overview"] = df["overview"].fillna("Overview not available")
df["genre"] = df["genre"].fillna("Unknown")
df["poster_url"] = df["poster_url"].fillna("Poster not available")
df["original_language"] = df["original_language"].fillna("unknown")

for column in ["popularity", "vote_count", "vote_average"]:
    df[column] = df[column].fillna(df[column].median())

df["vote_count"] = df["vote_count"].round().astype(int)
df["vote_average"] = df["vote_average"].clip(0, 10)
df["popularity"] = df["popularity"].clip(lower=0)

print("Missing values after filling:", df.isna().sum().sum())

### Outlier Detection and Treatment

We use boxplots and IQR. Outliers are capped so extreme blockbuster values do not dominate normal charts.

In [ ]:
def detect_iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return q1, q3, lower, upper

def cap_outliers_iqr(df, columns):
    cleaned = df.copy()
    report = []
    for column in columns:
        cleaned[f"{column}_original"] = cleaned[column]
        q1, q3, lower, upper = detect_iqr_bounds(cleaned[column])
        outlier_count = ((cleaned[column] < lower) | (cleaned[column] > upper)).sum()
        cleaned[column] = cleaned[column].clip(lower=lower, upper=upper)
        report.append({"column": column, "lower_bound": lower, "upper_bound": upper, "outliers_before_capping": int(outlier_count)})
    return cleaned, pd.DataFrame(report)

df, outlier_report = cap_outliers_iqr(df, ["popularity", "vote_count", "vote_average"])
display(outlier_report)

**Expected outlier report from generated project:**

| column       |      q1 |       q3 |   lower_bound |   upper_bound |   outliers_before_capping |   outliers_after_capping |
|:-------------|--------:|---------:|--------------:|--------------:|--------------------------:|-------------------------:|
| popularity   |  16.129 |   35.174 |        -12.44 |        63.743 |                      1048 |                        0 |
| vote_count   | 146     | 1376     |      -1699    |      3221     |                      1129 |                        0 |
| vote_average |   5.9   |    7.1   |          4.1  |         8.9   |                       243 |                        0 |

## 7. Data Processing and Feature Engineering

Create useful columns such as release year, decade, primary genre, language name, movie age, rating category, and weighted score.

In [ ]:
language_map = {
    "en": "English", "ja": "Japanese", "es": "Spanish", "fr": "French", "ko": "Korean",
    "zh": "Chinese", "cn": "Chinese", "it": "Italian", "ru": "Russian", "de": "German",
    "pt": "Portuguese", "hi": "Hindi"
}

df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
df["release_month_name"] = df["release_date"].dt.month_name()
df["release_decade"] = (df["release_year"] // 10 * 10).astype(int).astype(str) + "s"
df["genre_list"] = df["genre"].astype("string").str.split(",")
df["primary_genre"] = df["genre_list"].apply(lambda genres: genres[0].strip() if isinstance(genres, list) and genres else "Unknown")
df["genre_count"] = df["genre_list"].apply(lambda genres: len(genres) if isinstance(genres, list) else 0)
df["overview_word_count"] = df["overview"].astype("string").str.split().str.len().fillna(0).astype(int)
df["language_name"] = df["original_language"].astype("string").str.lower().map(language_map).fillna(df["original_language"].astype("string").str.upper())
df["is_english"] = np.where(df["original_language"].astype("string").str.lower() == "en", "English", "Non-English")
df["movie_age"] = 2026 - df["release_year"]
df["is_recent"] = np.where(df["release_year"] >= 2015, "Recent Movie", "Older Movie")
df["rating_category"] = pd.cut(df["vote_average"], bins=[-0.01, 4.99, 6.49, 7.49, 10], labels=["Low Rated", "Average", "Good", "Excellent"]).astype("string")
df["popularity_level"] = pd.qcut(df["popularity"].rank(method="first"), q=4, labels=["Low", "Medium", "High", "Very High"]).astype("string")
df["weighted_score"] = ((df["vote_average"] * np.log1p(df["vote_count"])) / np.log1p(df["vote_count"].max())).round(3)
display(df.head())

### Validate Cleaned Data

Check that the cleaned dataset has no missing values, no duplicate title/date records, and valid numeric ranges.

In [ ]:
validation = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "missing_values_total": df.isna().sum().sum(),
    "duplicate_rows": df.drop(columns=["genre_list"], errors="ignore").duplicated().sum(),
    "invalid_vote_average_rows": ((df["vote_average"] < 0) | (df["vote_average"] > 10)).sum(),
    "negative_popularity_rows": (df["popularity"] < 0).sum(),
}
validation

## 8. Exploratory Data Analysis

EDA means exploring the dataset to find patterns before giving final conclusions.

In [ ]:
display(df[["popularity", "vote_count", "vote_average", "weighted_score", "genre_count", "overview_word_count", "movie_age"]].describe().T)
display(df[["language_name", "primary_genre", "rating_category", "popularity_level"]].describe().T)

In [ ]:
genre_df = df[["title", "release_year", "popularity", "vote_count", "vote_average", "weighted_score", "genre_list"]].explode("genre_list").copy()
genre_df["genre_name"] = genre_df["genre_list"].astype("string").str.strip()

genre_summary = genre_df.groupby("genre_name", as_index=False).agg(
    movie_count=("title", "count"),
    avg_popularity=("popularity", "mean"),
    avg_rating=("vote_average", "mean"),
).sort_values("movie_count", ascending=False)

display(genre_summary.head(10))

## 9. Visualizations

The charts below show the main patterns found in the cleaned dataset.

### Visualization 1: Missing Values Before Cleaning

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/01_missing_values_before_cleaning.png'))

**Interpretation:** The raw movie dataset has a small number of missing values in title, overview, language, genre, poster URL, and numeric rating fields. These must be handled before analysis.

### Visualization 2: Outlier Detection Before Treatment

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/02_outlier_boxplots_before_treatment.png'))

**Interpretation:** Popularity and vote count contain very large blockbuster values. IQR capping creates stable analysis columns while preserving original values separately.

### Visualization 3: Top 10 Movies by Popularity

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/03_top_10_movies_by_popularity.png'))

**Interpretation:** The most popular movies are the highest-attention titles in the dataset. These films often represent strong public interest or platform visibility.

### Visualization 4: Top 10 Movies by Vote Count

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/04_top_10_movies_by_vote_count.png'))

**Interpretation:** Vote count shows audience engagement. Movies with many votes have stronger evidence behind their rating than movies with only a few votes.

### Visualization 5: Top Rated Movies with High Vote Count

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/05_top_rated_qualified_movies.png'))

**Interpretation:** Filtering by vote count avoids ranking movies highly just because very few people rated them. This produces a more reliable top-rated list.

### Visualization 6: Top 15 Genres by Movie Count

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/06_top_15_genres_by_count.png'))

**Interpretation:** Genre count shows which types of movies appear most often. Drama, comedy, action, and thriller-type genres often dominate large movie databases.

### Visualization 7: Average Popularity by Genre

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/07_average_popularity_by_genre.png'))

**Interpretation:** This chart shows which genres attract higher average attention. It is useful for understanding audience demand by content type.

### Visualization 8: Top Languages by Movie Count

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/08_top_languages_by_movie_count.png'))

**Interpretation:** Language distribution shows dataset coverage. English movies dominate, while Japanese, Spanish, French, Korean, and other languages add international variety.

### Visualization 9: Movie Releases by Year

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/09_movie_releases_by_year.png'))

**Interpretation:** Release trends show how the dataset changes over time. Recent years contain many movies because modern movie databases usually have richer coverage.

### Visualization 10: Movie Count by Decade

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/10_movie_count_by_decade.png'))

**Interpretation:** Decade analysis summarizes long-term coverage. It helps compare older cinema representation with modern releases.

### Visualization 11: Rating Distribution

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/11_rating_distribution_histogram.png'))

**Interpretation:** Most movies cluster around middle-to-good ratings. Very low and perfect ratings are less common after cleaning.

### Visualization 12: Popularity Distribution

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/12_popularity_distribution_histogram.png'))

**Interpretation:** Most movies have modest popularity, while a smaller group receives much higher attention. Capping makes the distribution easier to study.

### Visualization 13: Vote Count vs Popularity

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/13_vote_count_vs_popularity_scatter.png'))

**Interpretation:** Popular movies often receive more votes, but the relationship is not perfect. Some movies are highly rated or voted without being the most popular at the moment.

### Visualization 14: Overview Length vs Rating

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/14_overview_length_vs_rating_scatter.png'))

**Interpretation:** Overview length does not strongly determine rating, but this chart checks whether richer descriptions are associated with different rating patterns.

### Visualization 15: Rating Category Share

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/15_rating_category_pie_chart.png'))

**Interpretation:** The pie chart groups movies into simple rating bands. This makes it easy to explain the overall quality mix of the dataset.

### Visualization 16: Rating by Primary Genre

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/16_rating_by_primary_genre_boxplot.png'))

**Interpretation:** Boxplots compare rating ranges across genres. They show whether some genres tend to have higher median ratings or more variation.

### Visualization 17: Correlation Heatmap

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/17_correlation_heatmap.png'))

**Interpretation:** The heatmap shows relationships between numeric features. Popularity and vote count usually move together, while rating behaves differently.

### Visualization 18: Pairplot of Key Movie Metrics

In [ ]:
from IPython.display import Image, display

display(Image(filename='../outputs/charts/18_pairplot_key_movie_metrics.png'))

**Interpretation:** The pairplot gives a compact view of multiple relationships at once. It is useful for spotting clusters, patterns, and unusual movies.

## 10. Insights

- The cleaned dataset contains 9,827 valid movies across 27 columns.
- Spider-Man: No Way Home has the highest popularity score.
- Inception has the highest vote count.
- Dilwale Dulhania Le Jayenge stands out after filtering for movies with strong vote counts.
- Drama is the most common genre in the dataset.
- History has the highest average rating among genres with enough records for comparison.
- English is the dominant language, covering 77.0% of the cleaned dataset.
- The 2010s contain the largest number of movies in this dataset.
- Popularity and rating measure different things. A movie can be widely watched without being the highest rated.

## 11. Recommendations

- Use weighted score when ranking movies so both rating and vote count are considered.
- Compare genres separately because audience behavior changes by content type.
- Consider release year when comparing older and newer movies.
- Use language analysis when studying international content demand.
- Use original popularity and vote-count values for top-movie lists, but capped values for cleaner statistical charts.

## 12. Conclusion

This project cleaned and analyzed the MovieDB dataset from start to finish. The raw CSV was loaded safely, columns were standardized, text was cleaned, dates and numbers were converted, unusable rows were removed, missing values were handled, outliers were treated with IQR, and useful features were created.

The analysis shows that movie popularity, vote count, rating, genre, language, and release year all tell different parts of the movie-data story. The final cleaned dataset is ready for dashboarding, reporting, portfolio submission, and future recommendation-system projects.

## 13. Export Cleaned Dataset

In [ ]:
output_path = PROJECT_ROOT / "data" / "processed" / "mymoviedb_cleaned.csv"
df.to_csv(output_path, index=False)
print("Cleaned dataset exported to:", output_path)

## Project Summary

- Project title: MovieDB Data Cleaning and Visualization Project
- Raw dataset shape: `(9837, 9)`
- Cleaned dataset shape: `(9827, 27)`
- Cleaned movies: `9,827`
- Average rating: `6.44`
- Top popular movie: `Spider-Man: No Way Home`
- Top genre: `Drama`
- Top language: `English`
- Dashboard: `outputs/reports/movie_dashboard.html`